# Análisis de Datos, Minería y Recomendación Prescriptiva
**Equipo:** Calidad de Datos (Carlo Kiliano Ferrera, José Julian Pérez, Aldo Sebastián Altamirano)
**Curso:** Calidad y Preprocesamiento de Datos · Licenciatura en Ciencia de Datos · IIMAS UNAM CU · 2026-2
**Framework:** DAMA-DMBOK · Fase *Analyze*

---

## Objetivo del Notebook
Consolidar la etapa analítica y predictiva del proyecto extrayendo *insights* de los datos maestros limpios y fusionados. Este análisis se divide en tres bloques principales:

1. **Consultas Multi-fuente y Análisis Exploratorio Visual (EDA):** 10 consultas complejas cruzando calidad de agua, infraestructura, demografía y morbilidad con gráficos a nivel *Senior Data Scientist*.
2. **Modelo Predictivo (Machine Learning):** Entrenamiento de un modelo `XGBoost` para pronosticar el número de fugas en el próximo semestre usando un esquema de validación temporal (Train: 2018-2022, Val: 2023, Test: 2024).
3. **Sistema de Recomendación Prescriptivo:** Cálculo del *Score de Intervención* que cruza el riesgo de fugas futuro con el Índice de Vulnerabilidad Sanitaria (IVS), optimizando la asignación de un presupuesto limitado.

## 0. Configuración del Entorno y Carga de Datos

In [2]:
import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Configuraciones visuales
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 100)

In [3]:
def get_project_root(marker: str = "README.md") -> Path:
    """Sube el árbol de directorios hasta hallar el archivo `marker`."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

PROJECT_ROOT   = get_project_root()
DATOS_MAESTROS = PROJECT_ROOT / "datos" / "datos_maestros"

print(f"📂 PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"📂 DATOS_MAESTROS : {DATOS_MAESTROS}")

# Carga de tablas maestras
df_maestra = pd.read_csv(DATOS_MAESTROS / "maestra_colonia.csv", encoding="utf-8-sig")
df_semestre = pd.read_csv(DATOS_MAESTROS / "maestra_colonia_semestre.csv", encoding="utf-8-sig")
df_anio = pd.read_csv(DATOS_MAESTROS / "maestra_colonia_anio.csv", encoding="utf-8-sig")

print(f"\n✅ Maestra Snapshot cargada: {df_maestra.shape}")
print(f"✅ Maestra Semestral cargada: {df_semestre.shape}")

FileNotFoundError: No se encontró la raíz del proyecto.

## 1. Consultas Multi-fuente y Visualizaciones

Extracción de insights a través del cruce de dimensiones: SACMEX (Fugas) + CONEVAL (Pobreza) + CONAGUA (Calidad) + SSA (Morbilidad) + INEGI (Infraestructura).

In [ ]:
# Consulta 1: Dispersión - Densidad Poblacional vs Tasa de Fugas Totales por 10k Hab
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_maestra, x='densidad_pob_por_vivienda', y='fugas_por_10k_hab_total', 
                hue='nom_alcaldia', alpha=0.7, legend=False)
plt.title('C1: Densidad Poblacional vs Tasa de Fugas Históricas (por 10k Hab)')
plt.xlabel('Densidad Poblacional (Habitantes / Vivienda) [INEGI]')
plt.ylabel('Tasa de Fugas (Log Scale) [SACMEX]')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 2: Heatmap de Correlación Global - Fugas, Pobreza, Infraestructura y Morbilidad
cols_corr = ['fugas_por_10k_hab_total', 'pobreza_pct_promedio', 'pct_drenaje', 
             'pct_excede_nom_local', 'tasa_morbilidad_estimada_por_100k', 'antiguedad_red_proxy', 'IVS']
corr_matrix = df_maestra[cols_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('C2: Correlación Multi-fuente (SACMEX, CONEVAL, INEGI, CONAGUA, SSA)')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 3: Boxplot - Distribución del IVS (Índice de Vulnerabilidad Sanitaria) por Alcaldía
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_maestra, x='nom_alcaldia', y='IVS', palette='Set3')
plt.xticks(rotation=45, ha='right')
plt.title('C3: Distribución de la Vulnerabilidad Sanitaria Hídrica por Alcaldía')
plt.xlabel('')
plt.ylabel('IVS (Score 0-1)')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 4: Serie de Tiempo - Tendencia Histórica Semestral de Fugas (2018-2024)
tendencia_fugas = df_semestre.groupby('periodo_label')['n_fugas'].sum().reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=tendencia_fugas, x='periodo_label', y='n_fugas', marker='o', linewidth=2.5, color='#2980b9')
plt.title('C4: Evolución Semestral de Fugas Reportadas en CDMX (2018 - 2024)')
plt.xlabel('Semestre')
plt.ylabel('Volumen Total de Fugas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 5: Barras Apiladas - Composición Promedio del IVS por Alcaldía
ivs_comp = df_maestra.groupby('nom_alcaldia')[['pobreza_norm_ivs', 'morbi_norm_ivs']].mean().sort_values('pobreza_norm_ivs')

ivs_comp.plot(kind='bar', stacked=True, figsize=(14, 6), colormap='viridis', alpha=0.85)
plt.title('C5: Composición del IVS (Carencias Sociales CONEVAL vs Riesgo de Morbilidad SSA)')
plt.xlabel('')
plt.ylabel('Score Promedio Normalizado')
plt.legend(['Vulnerabilidad por Pobreza', 'Vulnerabilidad por Morbilidad Hídrica'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 6: Hexbin Plot - Antigüedad de la Red vs Porcentaje de Pobreza
plt.figure(figsize=(9, 6))
plt.hexbin(df_maestra['pobreza_pct_promedio'], df_maestra['antiguedad_red_proxy'], gridsize=25, cmap='Blues', mincnt=1)
cb = plt.colorbar(label='Densidad de Colonias')
plt.title('C6: Concentración Geoespacial: Antigüedad de Red Hídrica vs Pobreza')
plt.xlabel('Pobreza (% Promedio) [CONEVAL]')
plt.ylabel('Proxy de Antigüedad de Red [INEGI/Calculado]')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 7: Violinplot - Tiempos de Respuesta (Lag) de SACMEX según Carencias de Calidad de Espacios
df_maestra['carencia_espacios_binned'] = pd.qcut(df_maestra['carencia_calidad_espacios_pct'], q=3, labels=['Baja', 'Media', 'Alta'])

plt.figure(figsize=(10, 6))
sns.violinplot(data=df_maestra[df_maestra['lag_promedio_dias'] < 30], 
               x='carencia_espacios_binned', y='lag_promedio_dias', palette='flare')
plt.title('C7: Tiempos de Atención a Fugas SACMEX vs Deficiencias de Vivienda (CONEVAL)')
plt.xlabel('Nivel de Carencia en Calidad de Espacios de Vivienda')
plt.ylabel('Lag Promedio de Reparación (Días)')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 8: Barplot - Top 15 Colonias con mayor Incidencia de Fugas (Total Histórico)
top_fugas = df_maestra.nlargest(15, 'n_fugas_total')

plt.figure(figsize=(12, 6))
sns.barplot(data=top_fugas, y='nom_colonia', x='n_fugas_total', palette='Reds_r')
plt.title('C8: Top 15 Colonias con Mayor Volumen Acumulado de Fugas (2018-2024)')
plt.xlabel('Número Total de Fugas Registradas')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 9: Mapa de Calor Geoespacial - Morbilidad Estimada en la CDMX
# Se utilizan las coordenadas (centroide_lon, centroide_lat) como proxy geográfico
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_maestra['centroide_lon'], df_maestra['centroide_lat'], 
                      c=df_maestra['tasa_morbilidad_estimada_por_100k'], 
                      cmap='magma', s=df_maestra['pob_colonia']/200, alpha=0.6, edgecolors='none')
plt.colorbar(scatter, label='Tasa de Morbilidad (x 100k hab)', shrink=0.7)
plt.title('C9: Mapeo de Focos Rojos Sanitarios por Enfermedades Hídricas (Tamaño = Población)')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Consulta 10: Jointplot KDE - Distancia a la Planta Potabilizadora vs Porcentaje de Excedencia NOM-127
g = sns.jointplot(data=df_maestra, x='dist_planta_km', y='pct_excede_nom_local', 
                  kind="kde", cmap="rocket_r", fill=True, height=7, space=0)
g.fig.suptitle('C10: Relación entre Distancia a Planta Potabilizadora y Calidad del Agua', y=1.02)
g.set_axis_labels('Distancia a Planta Potabilizadora (km)', '% Mediciones que exceden NOM-127')
plt.show()

## 2. Modelo Predictivo (Machine Learning) - XGBoost

**Objetivo de Negocio:** Anticipar fallos en la red para pasar de una gestión reactiva a una proactiva. Predeciremos el número de fugas semestrales (`n_fugas`) por colonia.

**Esquema de Validación Temporal Estricto:**
Dado que logramos procesar en la Fusión v7 el histórico completo de SACMEX (2018-2024), mantendremos el diseño prometido en el README sin sesgos de *data leakage*:
*   **Entrenamiento (Train):** Datos desde 2018 hasta 2022.
*   **Validación (Val):** Datos de 2023 (utilizado para Early Stopping).
*   **Prueba (Test):** Datos de 2024 (evaluación de vida real y base para el recomendador).

In [ ]:
# 2.1 Preparación del Dataset Semestral
features = [
    'pob_colonia', 'pct_aguadv', 'pct_drenaje', 'densidad_pob_por_vivienda',
    'pobreza_pct_promedio', 'ingreso_lpi_pct_promedio', 'dist_sitio_km',
    'pct_excede_nom_local', 'factor_riesgo_morbilidad', 'antiguedad_red_proxy', 
    'dist_planta_km'
]
target = 'n_fugas'

# Filtrar nulos en features de la tabla semestral
df_ml = df_semestre.dropna(subset=features + [target]).copy()

# 2.2 Validación Temporal (Train/Val/Test Split)
df_train = df_ml[df_ml['anio'] <= 2022]
df_val   = df_ml[df_ml['anio'] == 2023]
df_test  = df_ml[df_ml['anio'] == 2024]

X_train, y_train = df_train[features], df_train[target]
X_val, y_val     = df_val[features], df_val[target]
X_test, y_test   = df_test[features], df_test[target]

print(f"🔹 Muestras Entrenamiento (2018-2022) : {X_train.shape[0]}")
print(f"🔹 Muestras Validación    (2023)      : {X_val.shape[0]}")
print(f"🔹 Muestras Prueba        (2024)      : {X_test.shape[0]}")

In [ ]:
# 2.3 Entrenamiento del Modelo XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# Ajuste con Early Stopping usando el set de Validación
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    early_stopping_rounds=50,
    verbose=False
)

# 2.4 Evaluación en Test (2024)
y_pred = xgb_model.predict(X_test)
y_pred = np.maximum(y_pred, 0) # Constreñir a 0 (no existen fugas negativas)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n================ RESULTADOS XGBOOST (TEST 2024) ================")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f} fugas por semestre")
print(f"MAE (Mean Absolute Error):      {mae:.2f} fugas por semestre")

# Inyectar la predicción en el DataFrame de Test para el motor de recomendación
df_test['prediccion_fugas_2024'] = y_pred

In [ ]:
# 2.5 Importancia de Características (Feature Importance)
plt.figure(figsize=(10, 6))
importances = pd.Series(xgb_model.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', color='#27ae60')
plt.title('Importancia de Variables (XGBoost) en la Predicción de Fugas')
plt.xlabel('Ganancia Relativa (Gain)')
plt.tight_layout()
plt.show()

## 3. Sistema de Recomendación Prescriptivo

En la administración pública los recursos son finitos. Para asesorar a SACMEX, creamos un ranking que balancea el mantenimiento de la red con el impacto en la equidad social.

**Score de Intervención** = `Riesgo de fuga pronosticado` × `Índice de Vulnerabilidad Sanitaria (IVS)`.

A continuación, **simularemos una intervención estratégica:** si SACMEX solo tiene presupuesto para renovar infraestructura en 30 colonias este semestre, ¿cuáles maximizan el impacto?

In [ ]:
# 3.1 Consolidación de Score por Colonia (Agregando semestres 2024)
df_recomendacion = df_test.groupby('id_colonia').agg({
    'nom_alcaldia': 'first',
    'nom_colonia': 'first',
    'prediccion_fugas_2024': 'sum', # Totalizamos las fugas predichas en 2024
    'IVS': 'first'
}).reset_index()

# 3.2 Normalización Min-Max del Riesgo Predictivo para equipararlo al IVS (ambos de 0 a 1)
max_fugas_pred = df_recomendacion['prediccion_fugas_2024'].max()
min_fugas_pred = df_recomendacion['prediccion_fugas_2024'].min()
df_recomendacion['riesgo_fuga_norm'] = (df_recomendacion['prediccion_fugas_2024'] - min_fugas_pred) / (max_fugas_pred - min_fugas_pred)

# 3.3 Cálculo del Score de Intervención Prescriptivo
df_recomendacion['Score_Intervencion_Predictivo'] = (df_recomendacion['riesgo_fuga_norm'] * df_recomendacion['IVS']).round(4)

# 3.4 SIMULACIÓN DE PRESUPUESTO: Top 30 Colonias Críticas
top_30_prioridad = df_recomendacion.sort_values(by='Score_Intervencion_Predictivo', ascending=False).head(30)

print("===================================================================================")
print("🎯 SIMULACIÓN: TOP 10 COLONIAS PARA PRIORIZACIÓN DE MANTENIMIENTO SACMEX")
print("===================================================================================")
display(top_30_prioridad[['nom_alcaldia', 'nom_colonia', 'prediccion_fugas_2024', 'IVS', 'Score_Intervencion_Predictivo']].head(10))

# 3.5 Visualización de la Recomendación Prescriptiva
plt.figure(figsize=(12, 8))
sns.barplot(data=top_30_prioridad.head(15), 
            x='Score_Intervencion_Predictivo', 
            y='nom_colonia',
            palette='YlOrRd_r')
plt.title('Top 15 Colonias: Máximo Impacto en Salud y Mantenimiento Hídrico')
plt.xlabel('Score de Intervención (XGBoost Fugas × IVS)')
plt.ylabel('')
plt.tight_layout()
plt.show()